# TUGAS MANDIRI
## Pertemuan 12 — Klasifikasi & ML Pipeline dengan Spark MLlib

| | |
|---|---|
| **Nama** | Emannuel Henji Pratama Putra |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **NPM** | 2505060012 |

---


## Menyiapkan Dataset

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


## SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12-2505060012") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/19 08:12:16 WARN Utils: Your hostname, mazedev-1701 resolves to a loopback address: 127.0.1.1; using 10.90.69.193 instead (on interface wlp2s0)
26/11/19 08:12:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 08:12:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/19 08:12:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


## Membuat DataFrame

In [ ]:
df = spark.read.csv("/mnt/MazedevUbuntu/tugas/praktikum/big-data/pertemuan-12/tugas12_ulasan.csv", header=True, inferSchema=True)
df.show()

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
|     Americano|Yogyakarta|       Pagi|  3|                10|             1|
|     Americano|Yogyakarta|      Siang|  2|                 5|             1|
|Kentang Goreng|Yogyakarta|      Malam|  5|                16|             0|
|    Roti Bakar|  Semarang|       Pagi|  5|                21|             0|
|     Kopi Susu|  Semarang|       Sore|  1|                23|  

## A. Membangun ML Pipeline

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Tahap 1-3: StringIndexer untuk setiap kolom kategorikal
idx_nama_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu_pesan = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Tahap 4: OneHotEncoder — mengubah ketiga kolom indeks menjadi representasi biner sekaligus
ohe = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

# Tahap 5: VectorAssembler — menggabungkan seluruh fitur menjadi satu kolom "features"
assembler = VectorAssembler(
    inputCols=["qty", "waktu_tunggu_menit", "nama_menu_vec", "cabang_vec", "waktu_pesan_vec"],
    outputCol="features"
)

# Tahap 6: Algoritma klasifikasi
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

print("Keenam tahap pipeline berhasil didefinisikan (belum dijalankan).")

Keenam tahap pipeline berhasil didefinisikan (belum dijalankan).


## B. Pelatihan & Evaluasi

In [ ]:
# Pipeline merangkai SELURUH tahap menjadi satu alur kerja
pipeline = Pipeline(stages=[idx_nama_menu, idx_cabang, idx_waktu_pesan, ohe, assembler, lr])

# Split data SEBELUM fit
train, test = df.randomSplit([0.8, 0.2], seed=77)

# fit() SEKALI SAJA menjalankan seluruh 6 tahap secara berurutan pada data training
model = pipeline.fit(train)

print("Pipeline berhasil dilatih!")
print(f"Data training: {train.count()} | Data testing: {test.count()}")

Pipeline berhasil dilatih!
Data training: 2411 | Data testing: 589


In [ ]:
# transform() pada data testing juga otomatis menjalankan SELURUH tahap
# (indexing, encoding, assembling) sebelum akhirnya menghasilkan prediksi -- tidak ada risiko lupa satu tahap
predictions = pipeline_model_columns = model.transform(test)

predictions.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability").show(10, truncate=False)

+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |9                 |1             |1.0       |[0.3058092647053278,0.6941907352946721] |
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414356,0.37461208245856437]|
|Americano|Magelang|Malam      |2  |2                 |1             |1.0       |[0.08601505227726759,0.9139849477227324]|
|Americano|Magelang|Malam      |3  |4                 |1             |1.0       |[0.12916081752315708,0.8708391824768429]|
|Americano|Magelang|Malam      |3  |13                |0             |0.0       |[0.5224849058902372,0.4775150941097628] |
|Americano|Magel

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Accuracy -- persentase prediksi yang benar secara keseluruhan
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
accuracy = ev_acc.evaluate(predictions)

# F1-score -- rata-rata harmonik precision & recall, lebih andal dibanding accuracy saat kelas tidak seimbang
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
f1 = ev_f1.evaluate(predictions)

# AUC-ROC -- mengukur kemampuan model membedakan kedua kelas di berbagai ambang batas (threshold)
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")
auc = ev_auc.evaluate(predictions)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"AUC-ROC  : {auc:.4f}")

Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650


## Model Pembanding

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Mendefinisikan algoritma BERBEDA -- Random Forest, sebuah ensemble dari banyak decision tree
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)

# Pipeline BARU -- lima tahap awal identik, hanya tahap terakhir yang diganti
pipeline_rf = Pipeline(stages=[idx_nama_menu, idx_cabang, idx_waktu_pesan, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)

accuracy_rf = ev_acc.evaluate(predictions_rf)
auc_rf = ev_auc.evaluate(predictions_rf)

print("=== Perbandingan Hasil ===")
print(f"Logistic Regression -> Accuracy: {accuracy:.4f} | AUC: {auc:.4f}")
print(f"Random Forest        -> Accuracy: {accuracy_rf:.4f} | AUC: {auc_rf:.4f}")

=== Perbandingan Hasil ===
Logistic Regression -> Accuracy: 0.7929 | AUC: 0.8650
Random Forest        -> Accuracy: 0.7963 | AUC: 0.8479


## Analisis Kesalahan Model

In [ ]:
predictions.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

# Interpretasi:
# label=0, prediction=0.0 -> True Negative  (benar memprediksi kelas 0)
# label=0, prediction=1.0 -> False Positive (salah, seharusnya 0 tapi diprediksi 1)
# label=1, prediction=0.0 -> False Negative (salah, seharusnya 1 tapi diprediksi 0)
# label=1, prediction=1.0 -> True Positive  (benar memprediksi kelas 1)

+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



## Kesimpulan Bisnis
Berdasarkan hasil evaluasi, model Logistic Regression layak dipilih sebagai model utama karena memiliki kemampuan pemisahan kelas terbaik (AUC-ROC = 0.8650). Dari sisi operasional "Kopi Nusantara", menjaga waktu_tunggu_menit tetap rendah (di bawah ambang ~12 menit) merupakan faktor kunci untuk mempertahankan peluang ulasan positif pelanggan, dan model ini dapat dimanfaatkan untuk mendeteksi lebih awal pesanan yang berisiko mendapatkan ulasan kurang baik.